In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

from sklearn.model_selection import train_test_split, validation_curve
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score,
                             confusion_matrix, roc_curve, auc)
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# Step 1: Load dataset
csv_path = "covid.csv"
if not os.path.exists(csv_path):
    raise FileNotFoundError(f"Dataset not found at: {csv_path}")

df = pd.read_csv(csv_path)

# Step 2: Handle categorical data (IMPORTANT)
df = pd.get_dummies(df, drop_first=True)

# Step 3: Define target (change if column name differs)
target_col = "covid"
if target_col not in df.columns:
    raise ValueError(f"Target column '{target_col}' not found. Available columns: {list(df.columns)}")

X = df.drop(target_col, axis=1)
y = df[target_col]

# Step 4: Convert to binary (needed for ROC)
y = (y == 1).astype(int)

# Step 5: Handle missing values (BEFORE scaling and splitting)
imputer = SimpleImputer(strategy="mean")
X = imputer.fit_transform(X)

# Step 5b: Train-test split (80% training) - DO THIS BEFORE SCALING
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

# Step 6: Feature scaling (FIT on training data only to avoid data leakage)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Step 7: Train Naive Bayes
model = GaussianNB()
model.fit(X_train, y_train)

# Step 8: Predictions
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

# Step 9: Metrics
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))

# Step 10: Confusion Matrix
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

# Step 11: ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr, label="AUC = %.2f" % roc_auc)
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()

# Step 12: Validation Curve (uses original unscaled data with cross-validation)
param_range = np.logspace(-9, 0, 10)

# Impute data for validation curve
X_imputed = imputer.transform(X)

train_scores, test_scores = validation_curve(
    GaussianNB(), X_imputed, y,
    param_name="var_smoothing",
    param_range=param_range,
    cv=5,
    scoring="accuracy"
)

train_mean = np.mean(train_scores, axis=1)
test_mean = np.mean(test_scores, axis=1)

plt.figure()
plt.plot(param_range, train_mean, label="Training Score")
plt.plot(param_range, test_mean, label="Validation Score")
plt.xscale("log")
plt.xlabel("var_smoothing")
plt.ylabel("Accuracy")
plt.title("Validation Curve (Naive Bayes)")
plt.legend()
plt.show()